## 1. Environment Setup

First, let's install all necessary libraries and set up our environment.

In [ ]:
%%capture
# Install required packages
!pip install -U datasets 
!pip install -U accelerate 
!pip install -U peft 
!pip install -U trl 
!pip install -U bitsandbytes
!pip install git+https://github.com/huggingface/transformers@v4.49.0-Gemma-3

In [ ]:
# Import necessary libraries
import os
import torch
import logging
from pathlib import Path
from transformers import (
    AutoTokenizer, 
    Gemma3ForConditionalGeneration,
    TrainingArguments,
    DataCollatorForLanguageModeling
)
from datasets import load_dataset
from trl import SFTTrainer
from peft import LoraConfig
from huggingface_hub import login

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print("Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f}GB")

In [ ]:
# Authentication setup
# Replace with your actual Hugging Face token
HF_TOKEN = "your_huggingface_token_here"

if HF_TOKEN != "your_huggingface_token_here":
    login(HF_TOKEN)
    print("✓ Hugging Face authentication successful")
else:
    print("⚠️  Please set your Hugging Face token above")

## 2. Model and Tokenizer Loading

Load the Gemma 3-4B-IT model and tokenizer with proper configuration.

In [ ]:
# Model configuration
MODEL_NAME = "google/gemma-3-4b-it"

print(f"Loading model: {MODEL_NAME}")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    padding_side="right",
    trust_remote_code=True
)

# Set pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("✓ Tokenizer loaded successfully")
print(f"Vocab size: {tokenizer.vocab_size:,}")
print(f"EOS token: '{tokenizer.eos_token}' (ID: {tokenizer.eos_token_id})")

In [ ]:
# Load model
model = Gemma3ForConditionalGeneration.from_pretrained(
    MODEL_NAME, 
    device_map="auto",
    torch_dtype=torch.float16,
    attn_implementation='eager',
    trust_remote_code=True
).eval()

print("✓ Model loaded successfully")
print(f"Model device: {next(model.parameters()).device}")
print(f"Model dtype: {next(model.parameters()).dtype}")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## 3. Dataset Processing

Load and format the financial reasoning dataset with proper prompt styling.

In [ ]:
# Define prompt templates
train_prompt_style = """
Below is an instruction that describes a task, paired with an input that provides further context. 
Write a response that appropriately completes the request. 
Before answering, think carefully about the question and create a step-by-step chain of thoughts to ensure a logical and accurate response.

### Question:
{}

### Response:
<think>
{}
</think>
{}
"""

inference_prompt_style = """Below is an instruction that describes a task, paired with an input that provides further context. 
Write a response that appropriately completes the request. 
Before answering, think carefully about the question and create a step-by-step chain of thoughts to ensure a logical and accurate response.

### Question:
{}

### Response:
<think>
{}
"""

print("✓ Prompt templates defined")

In [ ]:
# Dataset formatting function
def formatting_prompts_func(examples):
    """Format dataset examples into training prompts."""
    inputs = examples["Open-ended Verifiable Question"]
    complex_cots = examples["Complex_CoT"]
    outputs = examples["Response"]
    texts = []
    
    for question, cot, response in zip(inputs, complex_cots, outputs):
        # Ensure response ends with EOS token
        if not response.endswith(tokenizer.eos_token):
            response += tokenizer.eos_token
        
        # Format using training prompt style
        text = train_prompt_style.format(question, cot, response)
        texts.append(text)
    
    return {"text": texts}

print("✓ Formatting function defined")

In [ ]:
# Load dataset
DATASET_NAME = "TheFinAI/Fino1_Reasoning_Path_FinQA"
DATASET_SPLIT = "train[0:500]"  # Using first 500 samples for demo

print(f"Loading dataset: {DATASET_NAME}")
print(f"Split: {DATASET_SPLIT}")

dataset = load_dataset(
    DATASET_NAME, 
    split=DATASET_SPLIT,
    trust_remote_code=True
)

print(f"✓ Dataset loaded: {len(dataset)} samples")
print(f"Columns: {dataset.column_names}")

# Show sample
print("\n--- Sample Example ---")
sample = dataset[0]
print(f"Question: {sample['Open-ended Verifiable Question'][:100]}...")
print(f"Reasoning: {sample['Complex_CoT'][:100]}...")
print(f"Response: {sample['Response'][:100]}...")

In [ ]:
# Format dataset
formatted_dataset = dataset.map(
    formatting_prompts_func, 
    batched=True,
    desc="Formatting dataset"
)

print("✓ Dataset formatted successfully")
print(f"New columns: {formatted_dataset.column_names}")

# Show formatted example
print("\n--- Formatted Example ---")
print(formatted_dataset[0]["text"][:500] + "...")

In [ ]:
# Create data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # Causal LM, not masked LM
)

print("✓ Data collator created")

## 4. Pre-training Inference

Test the model's performance before fine-tuning to establish a baseline.

In [ ]:
def generate_response(model, tokenizer, question, reasoning_context="", max_new_tokens=1200):
    """Generate response for a given question."""
    prompt = inference_prompt_style.format(question, reasoning_context) + tokenizer.eos_token
    
    inputs = tokenizer(
        [prompt],
        return_tensors="pt"
    ).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_new_tokens=max_new_tokens,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True,
            temperature=0.7,
            do_sample=True
        )
    
    response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    # Extract just the response part
    response_part = response.split("### Response:")[1] if "### Response:" in response else response
    return response_part.strip()

print("✓ Response generation function defined")

In [ ]:
# Test pre-training performance
print("=== PRE-TRAINING INFERENCE ===")

test_question = dataset[0]['Open-ended Verifiable Question']
expected_cot = dataset[0]['Complex_CoT']
expected_response = dataset[0]['Response']

print(f"Question: {test_question}")
print("\n--- Expected Response ---")
print(f"Reasoning: {expected_cot}")
print(f"Answer: {expected_response}")

print("\n--- Pre-training Model Response ---")
pre_training_response = generate_response(model, tokenizer, test_question)
print(pre_training_response)

print(f"\nResponse length comparison:")
print(f"Expected: {len(expected_response)} chars")
print(f"Generated: {len(pre_training_response)} chars")

## 5. Training Setup

Configure LoRA (Low-Rank Adaptation) and training parameters for efficient fine-tuning.

In [ ]:
# LoRA Configuration
peft_config = LoraConfig(
    r=64,                                    # Rank of the LoRA update matrices
    lora_alpha=16,                           # Scaling factor for LoRA
    lora_dropout=0.05,                       # Dropout for regularization
    bias="none",                             # No bias reparameterization
    task_type="CAUSAL_LM",                   # Task type: Causal Language Modeling
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],  # Target modules for LoRA
)

print("✓ LoRA configuration defined")
print(f"LoRA rank: {peft_config.r}")
print(f"LoRA alpha: {peft_config.lora_alpha}")
print(f"Target modules: {peft_config.target_modules}")

In [ ]:
# Training Arguments
training_arguments = TrainingArguments(
    output_dir="./output",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=2,
    optim="paged_adamw_32bit",
    num_train_epochs=1,
    logging_steps=0.2,
    warmup_steps=10,
    logging_strategy="steps",
    learning_rate=2e-4,
    fp16=False,
    bf16=False,
    group_by_length=True,
    report_to="none"
)

print("✓ Training arguments configured")
print(f"Epochs: {training_arguments.num_train_epochs}")
print(f"Learning rate: {training_arguments.learning_rate}")
print(f"Batch size: {training_arguments.per_device_train_batch_size}")
print(f"Gradient accumulation: {training_arguments.gradient_accumulation_steps}")

In [ ]:
# Initialize SFTTrainer
trainer = SFTTrainer(
    model=model,
    args=training_arguments,
    train_dataset=formatted_dataset,
    peft_config=peft_config,
    data_collator=data_collator,
)

print("✓ SFTTrainer initialized")
print(f"Training dataset size: {len(formatted_dataset)}")

# Calculate training steps
total_steps = len(formatted_dataset) // (training_arguments.per_device_train_batch_size * training_arguments.gradient_accumulation_steps) * training_arguments.num_train_epochs
print(f"Estimated training steps: {total_steps}")

## 6. Model Training

Execute the fine-tuning process with progress monitoring.

In [ ]:
# Clear GPU cache before training
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("✓ GPU cache cleared")

print("\n=== STARTING TRAINING ===")
print("This may take a while depending on your hardware...")

# Start training
trainer_stats = trainer.train()

print("\n=== TRAINING COMPLETED ===")
print(f"Training loss: {trainer_stats.training_loss:.4f}")
print(f"Training runtime: {trainer_stats.metrics['train_runtime']:.2f} seconds")
print(f"Samples per second: {trainer_stats.metrics['train_samples_per_second']:.2f}")

## 7. Post-training Inference

Test the fine-tuned model and compare with pre-training performance.

In [ ]:
# Test post-training performance on the same question
print("=== POST-TRAINING INFERENCE ===")

print(f"Question: {test_question}")

print("\n--- Fine-tuned Model Response ---")
post_training_response = generate_response(model, tokenizer, test_question)
print(post_training_response)

print("\n=== COMPARISON ===")
print(f"Pre-training response length: {len(pre_training_response)} chars")
print(f"Post-training response length: {len(post_training_response)} chars")
print(f"Length improvement: {len(post_training_response) - len(pre_training_response)} chars")

In [ ]:
# Test on another sample
print("\n=== TESTING ON ANOTHER SAMPLE ===")

test_question_2 = dataset[10]['Open-ended Verifiable Question']
expected_response_2 = dataset[10]['Response']

print(f"Question 2: {test_question_2}")

print("\n--- Expected Response ---")
print(expected_response_2)

print("\n--- Fine-tuned Model Response ---")
response_2 = generate_response(model, tokenizer, test_question_2)
print(response_2)

## 8. Model Saving

Save the fine-tuned model and prepare for deployment.

In [ ]:
# Define save paths
LOCAL_MODEL_PATH = "./fine_tuned_gemma3_financial"
HUB_MODEL_NAME = "your-username/gemma-3-4b-fin-qa-reasoning"  # Change this to your desired name

print(f"Saving model locally to: {LOCAL_MODEL_PATH}")

# Save model and tokenizer locally
model.save_pretrained(LOCAL_MODEL_PATH)
tokenizer.save_pretrained(LOCAL_MODEL_PATH)

print("✓ Model and tokenizer saved locally")

# Verify saved files
saved_files = list(Path(LOCAL_MODEL_PATH).glob("*"))
print(f"Saved files: {[f.name for f in saved_files]}")

In [ ]:
# Optional: Push to Hugging Face Hub
# Uncomment and modify the following lines if you want to upload to Hugging Face

# print(f"Pushing model to Hugging Face Hub: {HUB_MODEL_NAME}")
# model.push_to_hub(HUB_MODEL_NAME)
# tokenizer.push_to_hub(HUB_MODEL_NAME)
# print("✓ Model pushed to Hugging Face Hub")

print("To upload to Hugging Face Hub:")
print(f"1. Change HUB_MODEL_NAME to your desired repository name")
print(f"2. Uncomment the push_to_hub lines above")
print(f"3. Run the cell")

## 9. Summary and Next Steps

Congratulations! You have successfully fine-tuned Gemma 3 for financial Q&A tasks.

In [ ]:
# Print final summary
print("="*60)
print("FINE-TUNING COMPLETE!")
print("="*60)

print(f"✓ Base model: {MODEL_NAME}")
print(f"✓ Dataset: {DATASET_NAME} ({len(dataset)} samples)")
print(f"✓ Training completed with loss: {trainer_stats.training_loss:.4f}")
print(f"✓ Model saved locally: {LOCAL_MODEL_PATH}")

print("\nKey Improvements Observed:")
print("• Enhanced reasoning capabilities")
print("• Structured step-by-step thinking")
print("• Better understanding of financial contexts")
print("• More detailed and accurate responses")

print("\nNext Steps:")
print("1. Test on more financial questions")
print("2. Evaluate on held-out test set")
print("3. Deploy for production use")
print("4. Consider additional fine-tuning on domain-specific data")

print("\nUsage Example:")
print(f"from transformers import AutoTokenizer, Gemma3ForConditionalGeneration")
print(f"model = Gemma3ForConditionalGeneration.from_pretrained('{LOCAL_MODEL_PATH}')")
print(f"tokenizer = AutoTokenizer.from_pretrained('{LOCAL_MODEL_PATH}')")

## Additional Resources

- **Gemma 3 Documentation**: [Google AI Blog](https://ai.google.dev/gemma)
- **FinQA Dataset**: [TheFinAI/Fino1_Reasoning_Path_FinQA](https://huggingface.co/datasets/TheFinAI/Fino1_Reasoning_Path_FinQA)
- **LoRA Paper**: [Low-Rank Adaptation of Large Language Models](https://arxiv.org/abs/2106.09685)
- **TRL Library**: [Transformer Reinforcement Learning](https://github.com/huggingface/trl)

---

*This notebook demonstrates the complete fine-tuning process. For production use, consider:*
- *Larger training datasets*
- *Proper validation splits*
- *Hyperparameter tuning*
- *Comprehensive evaluation metrics*